# 07 - Saving, Loading, Checkpoints & TorchScript (OPTIONAL)

> **This notebook is OPTIONAL.** It covers model persistence patterns and a brief introduction to TorchScript for deployment. TorchScript sections are guarded with `try/except`.

---

## Learning Objectives

- Save and load model weights using `torch.save` / `torch.load` with `state_dict`
- Save full training checkpoints (model + optimizer + epoch + metrics)
- Resume training from a checkpoint
- Implement a best-model saving pattern
- Understand TorchScript basics (`torch.jit.script`, `torch.jit.trace`)
- Know when ONNX export is relevant

## Prerequisites

- Notebooks 01-05 of this series
- Understanding of training loops (Notebook 05)
- Understanding of `nn.Module` and `state_dict` (Notebook 03)

## Table of Contents

1. [Saving & Loading state_dict](#1-saving--loading-state_dict)
2. [Full Checkpoint: Model + Optimizer + Epoch](#2-full-checkpoint-model--optimizer--epoch)
3. [Resuming Training from a Checkpoint](#3-resuming-training-from-a-checkpoint)
4. [Best Model Saving Pattern](#4-best-model-saving-pattern)
5. [TorchScript Basics](#5-torchscript-basics)
6. [ONNX Export (Concept)](#6-onnx-export-concept)
7. [Exercises](#7-exercises)
8. [Common Mistakes & Debugging Tips](#8-common-mistakes--debugging-tips)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import tempfile

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

# We will use a temp directory for all saved files
SAVE_DIR = tempfile.mkdtemp()
print(f"Save directory: {SAVE_DIR}")

In [ ]:
# Define a simple model we will reuse throughout
class SimpleNet(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=32, output_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
model = SimpleNet()
print(model)

## 1. Saving & Loading state_dict

The **recommended** way to save a model is to save its `state_dict` (an `OrderedDict` of parameter names to tensors).

| Approach | Pros | Cons |
|---|---|---|
| `torch.save(model.state_dict(), path)` | Portable, small file, safe | Must re-create model architecture |
| `torch.save(model, path)` | Saves everything | Fragile (uses pickle), breaks on refactor |

In [ ]:
# Save model weights
torch.manual_seed(42)
model_save = SimpleNet()

# Get predictions before saving (to verify later)
x_test = torch.randn(3, 10)
pred_original = model_save(x_test)
print(f"Original predictions:\n{pred_original}")

# Save
save_path = os.path.join(SAVE_DIR, 'model_weights.pth')
torch.save(model_save.state_dict(), save_path)
print(f"\nSaved to: {save_path}")
print(f"File size: {os.path.getsize(save_path):,} bytes")

In [ ]:
# Load model weights into a NEW model instance
model_load = SimpleNet()  # fresh model with random weights

# Before loading
pred_random = model_load(x_test)
print(f"Before loading (random):\n{pred_random}")

# Load saved weights
state_dict = torch.load(save_path, weights_only=True)
model_load.load_state_dict(state_dict)

# After loading
pred_loaded = model_load(x_test)
print(f"\nAfter loading:\n{pred_loaded}")
print(f"\nMatch original: {torch.allclose(pred_original, pred_loaded)}")

In [ ]:
# Inspect what is in a state_dict
print("Keys in state_dict:")
for key, tensor in state_dict.items():
    print(f"  {key:30s} shape={str(tensor.shape):15s} dtype={tensor.dtype}")

## 2. Full Checkpoint: Model + Optimizer + Epoch

For **resuming training**, you need more than just model weights:

| What to save | Why |
|---|---|
| `model.state_dict()` | Model weights |
| `optimizer.state_dict()` | Optimizer state (momentum, Adam betas) |
| `epoch` | Where to resume from |
| `loss` / metrics | For logging continuity |
| `scheduler.state_dict()` | Learning rate schedule state (if used) |

In [ ]:
# Train for a few epochs, then save a full checkpoint
torch.manual_seed(42)

model_ckpt = SimpleNet()
optimizer_ckpt = optim.Adam(model_ckpt.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Synthetic data
X_data = torch.randn(200, 10)
y_data = torch.randint(0, 3, (200,))

# Train for 10 epochs
train_losses = []
for epoch in range(10):
    model_ckpt.train()
    logits = model_ckpt(X_data)
    loss = loss_fn(logits, y_data)

    optimizer_ckpt.zero_grad()
    loss.backward()
    optimizer_ckpt.step()

    train_losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/10 | Loss: {loss.item():.4f}")

In [ ]:
# Save full checkpoint
checkpoint_path = os.path.join(SAVE_DIR, 'checkpoint.pth')

checkpoint = {
    'epoch': 10,
    'model_state_dict': model_ckpt.state_dict(),
    'optimizer_state_dict': optimizer_ckpt.state_dict(),
    'train_losses': train_losses,
    'best_loss': min(train_losses),
}

torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"File size: {os.path.getsize(checkpoint_path):,} bytes")
print(f"Saved at epoch {checkpoint['epoch']}, best loss: {checkpoint['best_loss']:.4f}")

## 3. Resuming Training from a Checkpoint

To resume training:
1. Create model and optimizer with the same architecture / hyperparameters
2. Load the checkpoint
3. Load `state_dict` for both model and optimizer
4. Set the starting epoch
5. Continue the training loop

In [ ]:
# Resume training
# Step 1: Create fresh model and optimizer
model_resume = SimpleNet()
optimizer_resume = optim.Adam(model_resume.parameters(), lr=0.01)

# Step 2: Load checkpoint
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Step 3: Load state dicts
model_resume.load_state_dict(checkpoint['model_state_dict'])
optimizer_resume.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch']
resumed_losses = checkpoint['train_losses']  # continue the loss history

print(f"Resumed from epoch {start_epoch}")
print(f"Previous losses: {[f'{l:.4f}' for l in resumed_losses]}")

In [ ]:
# Step 4 & 5: Continue training from where we left off
total_epochs = 20

for epoch in range(start_epoch, total_epochs):
    model_resume.train()
    logits = model_resume(X_data)
    loss = loss_fn(logits, y_data)

    optimizer_resume.zero_grad()
    loss.backward()
    optimizer_resume.step()

    resumed_losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{total_epochs} | Loss: {loss.item():.4f}")

print(f"\nTotal epochs trained: {len(resumed_losses)}")
print(f"Final loss: {resumed_losses[-1]:.4f}")

## 4. Best Model Saving Pattern

A common pattern during training:
- Track the best validation metric (e.g., lowest loss)
- Save the model only when it improves
- At the end, load the best model for evaluation

In [ ]:
# Best model saving during training
torch.manual_seed(42)

model_best = SimpleNet()
optimizer_best = optim.Adam(model_best.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Train / val split
X_train, X_val = X_data[:160], X_data[160:]
y_train, y_val = y_data[:160], y_data[160:]

best_val_loss = float('inf')
best_model_path = os.path.join(SAVE_DIR, 'best_model.pth')

num_epochs = 50
for epoch in range(num_epochs):
    # Train
    model_best.train()
    logits_t = model_best(X_train)
    loss_t = loss_fn(logits_t, y_train)
    optimizer_best.zero_grad()
    loss_t.backward()
    optimizer_best.step()

    # Validate
    model_best.eval()
    with torch.no_grad():
        logits_v = model_best(X_val)
        loss_v = loss_fn(logits_v, y_val)

    # Save if best
    if loss_v.item() < best_val_loss:
        best_val_loss = loss_v.item()
        torch.save(model_best.state_dict(), best_model_path)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:2d}: new best val loss = {best_val_loss:.4f} (saved)")

print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# Load the best model for evaluation
model_eval = SimpleNet()
model_eval.load_state_dict(torch.load(best_model_path, weights_only=True))
model_eval.eval()

with torch.no_grad():
    logits = model_eval(X_val)
    preds = logits.argmax(dim=1)
    acc = (preds == y_val).float().mean()

print(f"Best model val accuracy: {acc.item():.4f}")

## 5. TorchScript Basics

> This section is **optional** and guarded with `try/except`. TorchScript is useful for deployment but not required for training.

**TorchScript** converts a PyTorch model to an intermediate representation that can be:
- Serialized (saved without needing the original Python class definition)
- Run in C++ (no Python dependency)
- Optimized by the JIT compiler

Two conversion methods:

| Method | How it works | When to use |
|---|---|---|
| `torch.jit.trace(model, example_input)` | Records ops from one forward pass | Simple models, no control flow |
| `torch.jit.script(model)` | Compiles Python code to TorchScript | Models with `if`/`for`/control flow |

In [ ]:
# torch.jit.trace: record operations from one forward pass
try:
    torch.manual_seed(42)
    model_trace = SimpleNet()
    model_trace.eval()

    # Trace with example input
    example_input = torch.randn(1, 10)
    traced_model = torch.jit.trace(model_trace, example_input)

    # Compare outputs
    x = torch.randn(5, 10)
    out_original = model_trace(x)
    out_traced = traced_model(x)
    print(f"Outputs match: {torch.allclose(out_original, out_traced)}")

    # Save traced model (no Python class needed to load)
    traced_path = os.path.join(SAVE_DIR, 'traced_model.pt')
    traced_model.save(traced_path)
    print(f"Traced model saved to: {traced_path}")
    print(f"File size: {os.path.getsize(traced_path):,} bytes")

    # Load without needing the original class definition
    loaded_traced = torch.jit.load(traced_path)
    out_loaded = loaded_traced(x)
    print(f"Loaded traced model outputs match: {torch.allclose(out_original, out_loaded)}")

except Exception as e:
    print(f"TorchScript trace failed: {e}")
    print("This is expected in some environments. TorchScript is optional.")

In [ ]:
# torch.jit.script: compiles Python code (handles control flow)
try:
    class ConditionalNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(10, 20)
            self.fc2 = nn.Linear(20, 3)
            self.relu = nn.ReLU()

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.relu(self.fc1(x))
            x = self.fc2(x)
            # Control flow: trace would miss this
            if x.shape[0] > 1:
                x = x - x.mean(dim=0)  # batch normalization shortcut
            return x

    torch.manual_seed(42)
    cond_model = ConditionalNet()
    scripted_model = torch.jit.script(cond_model)

    x = torch.randn(5, 10)
    out_orig = cond_model(x)
    out_script = scripted_model(x)
    print(f"Outputs match: {torch.allclose(out_orig, out_script)}")

    # Save
    scripted_path = os.path.join(SAVE_DIR, 'scripted_model.pt')
    scripted_model.save(scripted_path)
    print(f"Scripted model saved to: {scripted_path}")

except Exception as e:
    print(f"TorchScript script failed: {e}")
    print("This is expected in some environments. TorchScript is optional.")

## 6. ONNX Export (Concept)

**ONNX** (Open Neural Network Exchange) is a standard format for representing ML models across frameworks.

- Export a PyTorch model to ONNX for use in TensorRT, ONNX Runtime, CoreML, etc.
- Useful for production deployment on diverse hardware

```python
# Conceptual example (not executed to avoid dependency issues)
import torch.onnx

model.eval()
dummy_input = torch.randn(1, 10)

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'},
                  'output': {0: 'batch_size'}},
)
```

**When to use ONNX:**
- Deploying to non-Python environments
- Using NVIDIA TensorRT for GPU inference optimization
- Serving models via ONNX Runtime (often faster than native PyTorch)

## 7. Exercises

### Exercise: Implement a CheckpointManager Class

Create a `CheckpointManager` class that:
1. Saves checkpoints to a specified directory
2. Tracks the best model based on a metric (e.g., validation loss)
3. Keeps only the last `max_to_keep` checkpoints (delete older ones)
4. Provides a `save(epoch, model, optimizer, metric)` method
5. Provides a `load_best()` method that returns the best checkpoint

In [ ]:
# TODO: Implement CheckpointManager

# class CheckpointManager:
#     def __init__(self, save_dir, max_to_keep=3):
#         ...
#
#     def save(self, epoch, model, optimizer, metric):
#         """Save a checkpoint. Delete old ones if over max_to_keep."""
#         ...
#
#     def load_best(self):
#         """Load and return the best checkpoint dict."""
#         ...
#
#     def load_latest(self):
#         """Load and return the most recent checkpoint dict."""
#         ...


# Test:
# torch.manual_seed(42)
# model = SimpleNet()
# optimizer = optim.Adam(model.parameters(), lr=0.01)
#
# mgr = CheckpointManager(os.path.join(SAVE_DIR, 'ckpt_mgr'), max_to_keep=3)
# mgr.save(epoch=1, model=model, optimizer=optimizer, metric=1.5)
# mgr.save(epoch=2, model=model, optimizer=optimizer, metric=1.2)
# mgr.save(epoch=3, model=model, optimizer=optimizer, metric=0.8)
# mgr.save(epoch=4, model=model, optimizer=optimizer, metric=0.9)
#
# best = mgr.load_best()
# print(f"Best checkpoint from epoch {best['epoch']}, metric={best['metric']}")

### Solution

In [ ]:
# Solution: CheckpointManager
class CheckpointManager:
    """Manages model checkpoints with best-model tracking and cleanup."""

    def __init__(self, save_dir, max_to_keep=3):
        self.save_dir = save_dir
        self.max_to_keep = max_to_keep
        self.best_metric = float('inf')
        self.best_path = None
        self.checkpoint_paths = []  # ordered list of saved paths
        os.makedirs(save_dir, exist_ok=True)

    def save(self, epoch, model, optimizer, metric):
        """Save a checkpoint. Delete old ones if over max_to_keep."""
        # Save checkpoint
        path = os.path.join(self.save_dir, f'checkpoint_epoch{epoch:04d}.pth')
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metric': metric,
        }
        torch.save(checkpoint, path)
        self.checkpoint_paths.append(path)

        # Track best model
        if metric < self.best_metric:
            self.best_metric = metric
            best_path = os.path.join(self.save_dir, 'best_model.pth')
            torch.save(checkpoint, best_path)
            self.best_path = best_path
            print(f"  Epoch {epoch}: new best (metric={metric:.4f})")

        # Cleanup old checkpoints
        while len(self.checkpoint_paths) > self.max_to_keep:
            old_path = self.checkpoint_paths.pop(0)
            if os.path.exists(old_path) and old_path != self.best_path:
                os.remove(old_path)

    def load_best(self):
        """Load and return the best checkpoint dict."""
        if self.best_path is None:
            raise FileNotFoundError("No best checkpoint found.")
        return torch.load(self.best_path, weights_only=False)

    def load_latest(self):
        """Load and return the most recent checkpoint dict."""
        if not self.checkpoint_paths:
            raise FileNotFoundError("No checkpoints found.")
        return torch.load(self.checkpoint_paths[-1], weights_only=False)


# Test
torch.manual_seed(42)
model_mgr = SimpleNet()
opt_mgr = optim.Adam(model_mgr.parameters(), lr=0.01)

mgr = CheckpointManager(os.path.join(SAVE_DIR, 'ckpt_mgr'), max_to_keep=3)

# Simulate training epochs with varying metrics
metrics = [1.5, 1.2, 0.8, 0.9, 0.7, 0.75]
for epoch, metric in enumerate(metrics, 1):
    mgr.save(epoch=epoch, model=model_mgr, optimizer=opt_mgr, metric=metric)

# Load best
best = mgr.load_best()
print(f"\nBest checkpoint: epoch={best['epoch']}, metric={best['metric']:.4f}")

# Load latest
latest = mgr.load_latest()
print(f"Latest checkpoint: epoch={latest['epoch']}, metric={latest['metric']:.4f}")

# Check that only max_to_keep checkpoints remain (plus best)
remaining = [f for f in os.listdir(os.path.join(SAVE_DIR, 'ckpt_mgr')) if f.startswith('checkpoint_')]
print(f"\nRemaining checkpoint files: {len(remaining)} (max_to_keep={mgr.max_to_keep})")
for f in sorted(remaining):
    print(f"  {f}")

## 8. Common Mistakes & Debugging Tips

1. **Saving `model` instead of `model.state_dict()`**: Saving the full model uses pickle, which breaks when you rename or move classes. Always save `state_dict()` for production.

2. **Forgetting to load optimizer state when resuming training**: Without the optimizer state, momentum and adaptive learning rate information is lost, causing a training spike after resuming.

3. **Loading weights on the wrong device**: When loading a model trained on GPU to CPU (or vice versa), use `map_location`:
   ```python
   torch.load('model.pth', map_location='cpu', weights_only=True)
   ```

4. **Mismatched architecture when loading**: `load_state_dict` will fail if the model architecture does not match the saved weights. Double-check that the model class has the same structure.

5. **Not calling `model.eval()` after loading for inference**: Dropout and BatchNorm behave differently in training vs eval mode. Always call `model.eval()` before inference.

6. **Overwriting the best model**: When saving the best model in a loop, make sure you save a separate copy (not just the latest). The model continues to train and may get worse.

7. **TorchScript `trace` missing control flow**: `torch.jit.trace` only records ops executed during the trace run. If/else branches not taken will be missing. Use `torch.jit.script` for models with control flow.

8. **Not using `weights_only=True` in `torch.load`**: For security, use `weights_only=True` when loading model weights. Use `weights_only=False` only for full checkpoints where you trust the source.

In [ ]:
# Clean up temporary files
import shutil
shutil.rmtree(SAVE_DIR, ignore_errors=True)
print("Temporary files cleaned up.")